# Average Temperature

There are several values that can be cached. The first that can be cached is the size of each bin in the x and y directions:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        for j in range(y_bins):
            x = (i + 0.5) * (x_bin_size)
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + 0.01 * x + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

Next, the value of `0.01 * x` is repeated many times in each loop:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + x_contribution + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

Next, the time contribution stays the same in each loop, so we can cache that value too:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + x_contribution + 0.005 * y + time_contribution + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

Next, the contribution from `y` is repeatedly calculated across different values of `x`. We can pre-compute all values and store them in a list so we simply need to access them:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

y_contributions = []
for j in range(y_bins):
    y = (j + 0.5) * (y_bin_size)
    y_contributions.append(0.005 * y)

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            total_temp = total_temp + x_contribution + y_contributions[j] + time_contribution + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

The final optimisation in this sample solution is not actually a caching operation. Instead, it is a mathematical one. For each of the `x_bins * y_bins` bins we add the value `time_contribution + 20`, then divide the whole final answer by `x_bins * y_bins`. A more efficient approach would be to remove the `+ time_contribution + 20` from the loop and , instead, add it it to the final value instead:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

y_contributions = []
for j in range(y_bins):
    y = (j + 0.5) * (y_bin_size)
    y_contributions.append(0.005 * y)

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            total_temp = total_temp + x_contribution + y_contributions[j] 
    print("Average Temperature: ", total_temp / (x_bins * y_bins) + time_contribution + 20)

%lprun -f average_temperature average_temperature(100)

# Morse Code

In this exercise, the `translate_word_to_morse` function will be called repeatedly with the same small number of words. This makes it a good place to add the `lru_cache` decorator to. The words that will be supplied as arguments are `North`, `South`, `East`, `West`, `0`, `10`, `20`, `30`, `40`, `50`, `60`, `70`, `80`, `90`, and `100`. There are fifteen of these words, so this is the minimum size of cache we should use.

In [ ]:
import cProfile
import random
from functools import lru_cache

@lru_cache(15)
def translate_word_to_morse(word):
    # This function translates a single word to morse code
    # Each letter is separated by a space
    morse_dict = {"a":".-", "b":"-...", "c":"-.-.", "d":"-..", "e":".", "f":"..-.", "g":"--.", "h":"....", "i":"..", "j":".---", "k":"-.-", "l":".-..", "m":"--", "n":"-.", "o":"---", "p":".--.", "q":"--.-", "r":".-.", "s":"...", "t":"-", "u":"..-", "v":"...-", "w":".--", "x":"-..-", "y":"-.--", "z":"--..", "1":".----", "2":"..---", "3":"...--", "4":"....-", "5":".....", "6":"-....", "7":"--...", "8":"---..", "9":"----.", "0":"-----"}
    morse_translation = ""
    for letter in word:
        morse_translation += morse_dict[letter] + " "
    return morse_translation.strip()

def translate_message_to_morse(message):
    # This function translates a full message to morse code
    # Each word is separated by three spaces
    morse_message = ""
    for word in message.split(" "):
        morse_message += translate_word_to_morse(word) + "   "
    return morse_message.strip()

def generate_random_message(length):
    # This first generates a message with length pairs of directions and distances
    # The direction will always be one of "north", "south", "east" and "west"
    # The distance will always be a multiple of 10 between 0 and 100
    directions = ["north", "east", "south", "west"]
    message = ""
    for i in range(length):
        direction = random.choice(directions)
        distance = 10 * random.randint(0, 10)
        message += direction + " " + str(distance) + " "
    return message

message = generate_random_message(10000)

cProfile.run('translate_message_to_morse(message)')